## Test Ollama Client

In [2]:
import sys, pathlib

# Add the project root (one level up from notebooks/) to sys.path
sys.path.insert(0, str(pathlib.Path().resolve().parent))

In [9]:
health_check = client.health_check()
print(f"health_check: {health_check}")


health_check: True


In [13]:
from src.adapters.ollama import OllamaAdapter



client = OllamaAdapter()



if not client.health_check():
    raise ConnectionError("Ollama is not reachable")

# chat(self, messages: list[dict], tools: list[dict])

resp= client.chat(
        messages=[
            {"role": "user", "content": "What is the capital of France?"},
            {"user": "What is the capital of France?"}
        ],
        tools=[],
)




In [14]:
resp

ModelResponse(content='The capital of France is Paris.', tool_calls=[], prompt_tokens=36, completion_tokens=8)

In [15]:
print(f"content: {resp.content}")
print(f"tool_calls: {resp.tool_calls}")
print(f"prompt_tokens: {resp.prompt_tokens}")
print(f"completion_tokens: {resp.completion_tokens}")

content: The capital of France is Paris.
tool_calls: []
prompt_tokens: 36
completion_tokens: 8


## Test ProductionToolRegistry

In [16]:
# --- Minimal fake tool for testing (no real Ollama needed) ---
from pydantic import BaseModel

class AddInput(BaseModel):
    a: int
    b: int

class FakeTool:
    name = "add"
    input_schema = AddInput

    def execute(self, validated: AddInput) -> str:
        return str(validated.a + validated.b)

    def get_ollama_schema(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": "Adds two numbers",
                "parameters": AddInput.model_json_schema(),
            },
        }

fake_tool = FakeTool()
print("FakeTool ready:", fake_tool.name)

FakeTool ready: add


In [17]:
from src.tool_assistant.tools.registry import ProductionToolRegistry

registry = ProductionToolRegistry()   # allowed_tools=None → allow all
registry.register_tool(fake_tool)

# __contains__
print("'add' in registry  :", "add" in registry)       # True
print("'sub' in registry  :", "sub" in registry)       # False

# get
print("get('add')         :", registry.get("add"))     # FakeTool
print("get('sub')         :", registry.get("sub"))     # None

'add' in registry  : True
'sub' in registry  : False
get('add')         : <__main__.FakeTool object at 0x000001C779601130>
get('sub')         : None


In [18]:
# get_schema — should return the Ollama schema for all registered tools
schemas = registry.get_schema()
print("schemas count:", len(schemas))   # 1
schemas

schemas count: 1


[{'type': 'function',
  'function': {'name': 'add',
   'description': 'Adds two numbers',
   'parameters': {'properties': {'a': {'title': 'A', 'type': 'integer'},
     'b': {'title': 'B', 'type': 'integer'}},
    'required': ['a', 'b'],
    'title': 'AddInput',
    'type': 'object'}}}]

In [19]:
# execute — happy path
result, success = registry.execute(
    tool_name="add",
    arguments={"a": 3, "b": 4},
    session_id="test-session",
    iteration=0,
)
print("result :", result)    # '7'
print("success:", success)   # True

result : 7
success: True


In [20]:
# execute — unknown tool
result, success = registry.execute(
    tool_name="unknown_tool",
    arguments={},
    session_id="test-session",
)
print("result :", result)    # Error: Unknown tool ...
print("success:", success)   # False

result : Error: Unknown tool 'unknown_tool'. Available: ['add']
success: False


In [21]:
# execute — validation error (wrong types)
result, success = registry.execute(
    tool_name="add",
    arguments={"a": "not-a-number", "b": 4},
    session_id="test-session",
)
print("result :", result)    # Error: Invalid arguments ...
print("success:", success)   # False

result : Error: Invalid arguments for add: a: Input should be a valid integer, unable to parse string as an integer
success: False


In [22]:
# execute — allowlist blocks a tool
restricted_registry = ProductionToolRegistry(allowed_tools={"multiply"})  # 'add' not allowed
restricted_registry.register_tool(fake_tool)

result, success = restricted_registry.execute(
    tool_name="add",
    arguments={"a": 3, "b": 4},
    session_id="test-session",
)
print("result :", result)    # Error: Tool 'add' is not in the allowed tools list
print("success:", success)   # False

result : Error: Tool 'add' is not in the allowed tools list
success: False


In [23]:
# call_logs — every execute() should write a log entry
print("log entries:", len(registry.call_logs))   # 3 (happy path + unknown + validation)
for log in registry.call_logs:
    print(f"  [{log.tool_name}] success={log.success} latency={log.latency_ms}ms")

log entries: 3
  [add] success=True latency=85.92ms
  [unknown_tool] success=False latency=0.0ms
  [add] success=False latency=12.01ms
